# ⚽ Smart Scouting System — Full Data Pipeline
### Sources: FBref (real stats) + Transfermarkt Kaggle CSVs
---

## 📦 Step 0 — Install Dependencies

In [ ]:
!pip install pandas requests beautifulsoup4 lxml tqdm soccerdata openpyxl

## 🗺️ Step 1 — League Mapping (Kaggle ID → FBref URL)
Maps every first-tier league in your Kaggle competitions.csv to its FBref stats page.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# LEAGUE MAPPING: Kaggle competition_id → FBref league URL
# ─────────────────────────────────────────────────────────────────
# Strategy:
#   - Big 5 + some extra → use soccerdata (cleaner API)
#   - All others         → direct BeautifulSoup scraping
# ─────────────────────────────────────────────────────────────────

LEAGUE_MAP = {

    # ── BIG 5 EUROPEAN (soccerdata supported) ──────────────────────
    "GB1":  {"name": "Premier League",        "fbref_id": "9",   "fbref_slug": "Premier-League",        "country": "England",     "method": "soccerdata", "sd_name": "ENG-Premier League"},
    "ES1":  {"name": "La Liga",               "fbref_id": "12",  "fbref_slug": "La-Liga",               "country": "Spain",       "method": "soccerdata", "sd_name": "ESP-La Liga"},
    "L1":   {"name": "Bundesliga",            "fbref_id": "20",  "fbref_slug": "Bundesliga",            "country": "Germany",     "method": "soccerdata", "sd_name": "GER-Bundesliga"},
    "IT1":  {"name": "Serie A",               "fbref_id": "11",  "fbref_slug": "Serie-A",               "country": "Italy",       "method": "soccerdata", "sd_name": "ITA-Serie A"},
    "FR1":  {"name": "Ligue 1",               "fbref_id": "13",  "fbref_slug": "Ligue-1",               "country": "France",      "method": "soccerdata", "sd_name": "FRA-Ligue 1"},

    # ── OTHER EUROPEAN LEAGUES (direct scrape) ─────────────────────
    "NL1":  {"name": "Eredivisie",            "fbref_id": "23",  "fbref_slug": "Eredivisie",            "country": "Netherlands", "method": "scrape"},
    "PO1":  {"name": "Primeira Liga",         "fbref_id": "32",  "fbref_slug": "Primeira-Liga",         "country": "Portugal",    "method": "scrape"},
    "BE1":  {"name": "Pro League",            "fbref_id": "37",  "fbref_slug": "Belgian-Pro-League",    "country": "Belgium",     "method": "scrape"},
    "TR1":  {"name": "Süper Lig",             "fbref_id": "26",  "fbref_slug": "Super-Lig",             "country": "Turkey",      "method": "scrape"},
    "GR1":  {"name": "Super League Greece",   "fbref_id": "27",  "fbref_slug": "Super-League-Greece",   "country": "Greece",      "method": "scrape"},
    "SC1":  {"name": "Scottish Premiership",  "fbref_id": "40",  "fbref_slug": "Scottish-Premiership",  "country": "Scotland",    "method": "scrape"},
    "A1":   {"name": "Austrian Bundesliga",   "fbref_id": "56",  "fbref_slug": "Austrian-Football-Bundesliga", "country": "Austria", "method": "scrape"},
    "C1":   {"name": "Swiss Super League",    "fbref_id": "57",  "fbref_slug": "Swiss-Super-League",    "country": "Switzerland", "method": "scrape"},
    "DK1":  {"name": "Danish Superliga",      "fbref_id": "50",  "fbref_slug": "Danish-Superliga",      "country": "Denmark",     "method": "scrape"},
    "SE1":  {"name": "Allsvenskan",           "fbref_id": "67",  "fbref_slug": "Allsvenskan",           "country": "Sweden",      "method": "scrape"},
    "NO1":  {"name": "Eliteserien",           "fbref_id": "68",  "fbref_slug": "Eliteserien",           "country": "Norway",      "method": "scrape"},
    "PL1":  {"name": "Ekstraklasa",           "fbref_id": "36",  "fbref_slug": "Ekstraklasa",           "country": "Poland",      "method": "scrape"},
    "RU1":  {"name": "Russian Premier League","fbref_id": "30",  "fbref_slug": "Russian-Premier-League","country": "Russia",      "method": "scrape"},
    "TS1":  {"name": "Czech First League",    "fbref_id": "66",  "fbref_slug": "Czech-First-League",    "country": "Czech Republic", "method": "scrape"},
    "RO1":  {"name": "Romanian Liga I",       "fbref_id": "47",  "fbref_slug": "Liga-I",                "country": "Romania",     "method": "scrape"},
    "SER1": {"name": "Serbian SuperLiga",     "fbref_id": "63",  "fbref_slug": "Serbian-SuperLiga",     "country": "Serbia",      "method": "scrape"},
    "KR1":  {"name": "HNL Croatia",           "fbref_id": "44",  "fbref_slug": "HNL",                   "country": "Croatia",     "method": "scrape"},
    "UKR1": {"name": "Ukrainian Premier League","fbref_id": "39","fbref_slug": "Ukrainian-Premier-League","country": "Ukraine", "method": "scrape"},

    # ── AMERICAS ───────────────────────────────────────────────────
    "MLS1": {"name": "MLS",                   "fbref_id": "22",  "fbref_slug": "Major-League-Soccer",   "country": "USA",         "method": "scrape"},
    "BRA1": {"name": "Brasileirao Serie A",   "fbref_id": "24",  "fbref_slug": "Serie-A",               "country": "Brazil",      "method": "scrape"},
    "ARG1": {"name": "Liga Profesional",      "fbref_id": "21",  "fbref_slug": "Primera-Division",      "country": "Argentina",   "method": "scrape"},
    "MEX1": {"name": "Liga MX",               "fbref_id": "31",  "fbref_slug": "Liga-MX",               "country": "Mexico",      "method": "scrape"},
    "COL1": {"name": "Liga BetPlay",          "fbref_id": "41",  "fbref_slug": "Categoria-Primera-A",   "country": "Colombia",    "method": "scrape"},

    # ── ASIA / PACIFIC ─────────────────────────────────────────────
    "JAP1": {"name": "J1 League",             "fbref_id": "25",  "fbref_slug": "J1-League",             "country": "Japan",       "method": "scrape"},
    "RSK1": {"name": "K League 1",            "fbref_id": "55",  "fbref_slug": "K-League-1",            "country": "South Korea", "method": "scrape"},
    "AUS1": {"name": "A-League Men",          "fbref_id": "53",  "fbref_slug": "A-League-Men",          "country": "Australia",   "method": "scrape"},

    # ── MIDDLE EAST ────────────────────────────────────────────────
    "SA1":  {"name": "Saudi Pro League",      "fbref_id": "70",  "fbref_slug": "Saudi-Professional-League", "country": "Saudi Arabia", "method": "scrape"},
}

print(f"✅ Total leagues mapped: {len(LEAGUE_MAP)}")
for k, v in LEAGUE_MAP.items():
    print(f"  [{k}] {v['name']:35s} ({v['country']}) → method: {v['method']}")

## 🔧 Step 2 — FBref Scraper (All Leagues)
Two methods:
- `soccerdata` for Big 5 (cleaner, more reliable)
- `BeautifulSoup + pandas` for all other leagues

In [ ]:
import requests
import pandas as pd
import time
import random
from bs4 import BeautifulSoup
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ── Rotating user agents to avoid FBref blocking ──
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/118.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/121.0",
]

# ── Stat types to scrape per league ──
# Each maps to an FBref table ID and URL path
STAT_TYPES = {
    "standard":  "stats",        # Goals, assists, xG, xA, minutes played
    "shooting":  "shooting",     # Shots, shots on target, xG per shot
    "passing":   "passing",      # Passes, progressive passes, key passes, xA
    "defense":   "defense",      # Tackles, interceptions, pressures, blocks
    "possession":"possession",   # Dribbles, carries, progressive carries
    "misc":      "misc",         # Fouls, aerials won, recoveries
}

def scrape_fbref_league(
    fbref_id: str,
    fbref_slug: str,
    season: str = "2023-2024",
    stat_type: str = "standard",
    delay: float = 4.0
) -> pd.DataFrame:
    """
    Scrape a specific stat table from FBref for any league.

    Args:
        fbref_id:   FBref numeric competition ID (e.g. '9' for Premier League)
        fbref_slug: FBref URL slug (e.g. 'Premier-League')
        season:     Season string (e.g. '2023-2024')
        stat_type:  One of: standard, shooting, passing, defense, possession, misc
        delay:      Seconds to wait between requests (respect rate limiting)

    Returns:
        DataFrame with player stats
    """
    stat_path = STAT_TYPES.get(stat_type, "stats")

    # FBref URL pattern for player stats
    url = (
        f"https://fbref.com/en/comps/{fbref_id}/{season}/{stat_path}/"
        f"players/{season}-{fbref_slug}-Stats"
    )

    headers = {"User-Agent": random.choice(USER_AGENTS)}

    try:
        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()

        # FBref tables often have multi-level headers — flatten them
        tables = pd.read_html(resp.text, header=[0, 1])

        if not tables:
            print(f"  ⚠️  No table found: {url}")
            return pd.DataFrame()

        df = tables[0].copy()

        # Flatten multi-level column headers
        df.columns = [
            f"{a}_{b}" if "Unnamed" not in str(a) else str(b)
            for a, b in df.columns
        ]

        # Drop repeated header rows that FBref embeds in the table
        if "Player" in df.columns:
            df = df[df["Player"] != "Player"].reset_index(drop=True)
        elif "Rk" in df.columns:
            df = df[df["Rk"] != "Rk"].reset_index(drop=True)

        # Tag source metadata
        df["fbref_season"]   = season
        df["fbref_stat_type"] = stat_type
        df["fbref_league_id"] = fbref_id
        df["fbref_slug"]     = fbref_slug

        time.sleep(delay)  # ← CRITICAL: respect FBref rate limiting
        return df

    except Exception as e:
        print(f"  ❌ Error scraping {url}: {e}")
        time.sleep(delay * 2)  # back off longer on error
        return pd.DataFrame()


print("✅ Scraper function defined")

## 🚀 Step 3 — Scrape Big 5 via soccerdata (Fast & Clean)

In [ ]:
import soccerdata as sd

BIG5_LEAGUES = [
    "ENG-Premier League",
    "ESP-La Liga",
    "GER-Bundesliga",
    "ITA-Serie A",
    "FRA-Ligue 1",
]

SEASON = "2023-2024"

fbref = sd.FBref(leagues=BIG5_LEAGUES, seasons=SEASON)

print("📥 Scraping Big 5 stats via soccerdata...")

big5_standard   = fbref.read_player_season_stats(stat_type="standard")
big5_shooting   = fbref.read_player_season_stats(stat_type="shooting")
big5_passing    = fbref.read_player_season_stats(stat_type="passing")
big5_defense    = fbref.read_player_season_stats(stat_type="defense")
big5_possession = fbref.read_player_season_stats(stat_type="possession")
big5_misc       = fbref.read_player_season_stats(stat_type="misc")

# Reset index to flatten MultiIndex
for df in [big5_standard, big5_shooting, big5_passing, big5_defense, big5_possession, big5_misc]:
    df.reset_index(inplace=True)

print(f"✅ Big 5 standard stats: {big5_standard.shape[0]:,} players")
print(f"   Columns: {list(big5_standard.columns[:10])}...")

## 🌍 Step 4 — Scrape All Other Leagues via BeautifulSoup

In [ ]:
# ── Only scrape non-Big5 leagues ──
NON_BIG5 = {
    k: v for k, v in LEAGUE_MAP.items()
    if v["method"] == "scrape"
}

all_leagues_dfs = {}

print(f"📥 Scraping {len(NON_BIG5)} leagues...")
print("⏱️  This will take ~15-25 minutes due to rate limiting.\n")

for comp_id, info in tqdm(NON_BIG5.items(), desc="Leagues"):
    league_dfs = []
    print(f"\n  🌍 {info['name']} ({info['country']})")

    for stat_type in STAT_TYPES.keys():
        df = scrape_fbref_league(
            fbref_id=info["fbref_id"],
            fbref_slug=info["fbref_slug"],
            season=SEASON,
            stat_type=stat_type,
            delay=4.0  # 4 second delay between each request
        )
        if not df.empty:
            df["kaggle_competition_id"] = comp_id
            df["league_name"]           = info["name"]
            df["league_country"]        = info["country"]
            league_dfs.append(df)
            print(f"    ✅ {stat_type}: {len(df)} players")
        else:
            print(f"    ⚠️  {stat_type}: empty")

    if league_dfs:
        all_leagues_dfs[comp_id] = league_dfs

print("\n✅ All leagues scraped!")

## 🔗 Step 5 — Merge All FBref Data Into One Master FBref DataFrame

In [ ]:
import re

def merge_stat_types(dfs_list: list) -> pd.DataFrame:
    """
    Given a list of DataFrames (one per stat type for same league),
    merge them all on Player + Squad + Season into one wide DataFrame.
    """
    if not dfs_list:
        return pd.DataFrame()

    # Identify player key columns (common across all stat types)
    key_cols = ["Player", "Squad", "Nation", "Pos", "Age",
                "fbref_season", "kaggle_competition_id",
                "league_name", "league_country"]

    result = None
    for df in dfs_list:
        # Keep only columns that exist
        available_keys = [c for c in key_cols if c in df.columns]
        stat_type      = df["fbref_stat_type"].iloc[0] if "fbref_stat_type" in df.columns else "unknown"

        # Rename non-key columns with stat type prefix to avoid clashes
        rename_map = {}
        for col in df.columns:
            if col not in available_keys + ["fbref_stat_type", "fbref_league_id", "fbref_slug", "Rk", "Matches"]:
                rename_map[col] = f"{stat_type}__{col}"
        df_renamed = df.rename(columns=rename_map)

        if result is None:
            result = df_renamed
        else:
            merge_on = [c for c in available_keys if c in result.columns and c in df_renamed.columns]
            result = result.merge(df_renamed, on=merge_on, how="outer", suffixes=("", f"_dup"))
            # Drop duplicate columns
            result = result[[c for c in result.columns if not c.endswith("_dup")]]

    return result


# ── Process non-Big5 leagues ──
non_big5_merged_parts = []
for comp_id, dfs_list in all_leagues_dfs.items():
    merged = merge_stat_types(dfs_list)
    if not merged.empty:
        non_big5_merged_parts.append(merged)
        print(f"  ✅ {comp_id}: {merged.shape[0]} players x {merged.shape[1]} cols")

non_big5_df = pd.concat(non_big5_merged_parts, ignore_index=True) if non_big5_merged_parts else pd.DataFrame()

# ── Process Big 5 leagues ──
big5_all = big5_standard.copy()
for df, suffix in [
    (big5_shooting,   "shooting"),
    (big5_passing,    "passing"),
    (big5_defense,    "defense"),
    (big5_possession, "possession"),
    (big5_misc,       "misc"),
]:
    merge_keys = [c for c in ["player","squad","season","league"] if c in big5_all.columns and c in df.columns]
    big5_all = big5_all.merge(df, on=merge_keys, how="left", suffixes=("", f"_{suffix}"))

# ── Stack Big 5 + non-Big5 ──
fbref_master = pd.concat([big5_all, non_big5_df], ignore_index=True)

print(f"\n✅ Total FBref master: {fbref_master.shape[0]:,} players x {fbref_master.shape[1]} columns")
fbref_master.to_csv("fbref_all_leagues.csv", index=False)
print("💾 Saved → fbref_all_leagues.csv")

## 📦 Step 6 — Load Your Transfermarkt Kaggle CSVs

In [ ]:
# ─────────────────────────────────────────────────────────
# UPDATE THIS PATH to where your CSV files are located
# ─────────────────────────────────────────────────────────
DATA_PATH = "./data/"  # ← change this

print("📥 Loading Transfermarkt Kaggle datasets...")

players        = pd.read_csv(f"{DATA_PATH}players.csv")
player_vals    = pd.read_csv(f"{DATA_PATH}player_valuations.csv")
transfers      = pd.read_csv(f"{DATA_PATH}transfers.csv")
appearances    = pd.read_csv(f"{DATA_PATH}appearances.csv")
game_events    = pd.read_csv(f"{DATA_PATH}game_events.csv")
game_lineups   = pd.read_csv(f"{DATA_PATH}game_lineups.csv")
games          = pd.read_csv(f"{DATA_PATH}games.csv")
clubs          = pd.read_csv(f"{DATA_PATH}clubs.csv")
competitions   = pd.read_csv(f"{DATA_PATH}competitions.csv")
club_games     = pd.read_csv(f"{DATA_PATH}club_games.csv")
nat_teams      = pd.read_csv(f"{DATA_PATH}national_teams.csv")
countries      = pd.read_csv(f"{DATA_PATH}countries.csv")

print(f"  players:        {len(players):>10,} rows")
print(f"  player_vals:    {len(player_vals):>10,} rows")
print(f"  transfers:      {len(transfers):>10,} rows")
print(f"  appearances:    {len(appearances):>10,} rows")
print(f"  game_events:    {len(game_events):>10,} rows")
print(f"  game_lineups:   {len(game_lineups):>10,} rows")
print(f"  games:          {len(games):>10,} rows")
print(f"\n✅ All datasets loaded")

## 🔨 Step 7 — Build Transfermarkt Feature Table

In [ ]:
# ── 7a. Latest market valuation per player ──
player_vals["date"] = pd.to_datetime(player_vals["date"], errors="coerce")
latest_val = (
    player_vals.sort_values("date")
    .groupby("player_id")
    .last()
    .reset_index()[["player_id", "market_value_in_eur", "date"]]
    .rename(columns={"market_value_in_eur": "latest_market_value_eur",
                     "date": "valuation_date"})
)

# ── 7b. Peak market value ever ──
peak_val = (
    player_vals.groupby("player_id")["market_value_in_eur"]
    .max()
    .reset_index()
    .rename(columns={"market_value_in_eur": "peak_market_value_eur"})
)

# ── 7c. Transfer history summary per player ──
transfer_summary = (
    transfers.groupby("player_id")
    .agg(
        total_transfers=("transfer_fee", "count"),
        total_fees_paid=("transfer_fee", "sum"),
        max_transfer_fee=("transfer_fee", "max"),
        last_transfer_date=("transfer_date", "max"),
        last_club_from=("from_club_name", "last"),
        last_club_to=("to_club_name", "last"),
    )
    .reset_index()
)

# ── 7d. Career stats from appearances (per player totals) ──
career_stats = (
    appearances.groupby("player_id")
    .agg(
        career_games=("appearance_id", "count"),
        career_goals=("goals", "sum"),
        career_assists=("assists", "sum"),
        career_yellow_cards=("yellow_cards", "sum"),
        career_red_cards=("red_cards", "sum"),
        career_minutes=("minutes_played", "sum"),
    )
    .reset_index()
)
# Add per-90 career metrics
career_stats["career_goals_per90"]   = (career_stats["career_goals"]   / career_stats["career_minutes"] * 90).round(3)
career_stats["career_assists_per90"] = (career_stats["career_assists"] / career_stats["career_minutes"] * 90).round(3)

# ── 7e. Goals and assists in last 12 months (form indicator) ──
appearances["date"] = pd.to_datetime(appearances["date"], errors="coerce")
cutoff = appearances["date"].max() - pd.DateOffset(months=12)
recent_apps = appearances[appearances["date"] >= cutoff]
recent_form = (
    recent_apps.groupby("player_id")
    .agg(
        recent_games=("appearance_id", "count"),
        recent_goals=("goals", "sum"),
        recent_assists=("assists", "sum"),
        recent_minutes=("minutes_played", "sum"),
    )
    .reset_index()
    .rename(columns=lambda x: f"{x}" if x == "player_id" else f"last12m_{x}")
)

# ── 7f. Club info for current club ──
club_info = clubs[["club_id","name","domestic_competition_id",
                   "total_market_value","squad_size","average_age"]].rename(
    columns={"name": "club_name", "total_market_value": "club_total_value"}
)

print("✅ Transfermarkt feature tables built:")
print(f"   latest_val:        {len(latest_val):,} players")
print(f"   transfer_summary:  {len(transfer_summary):,} players")
print(f"   career_stats:      {len(career_stats):,} players")
print(f"   recent_form:       {len(recent_form):,} players")

## 🧩 Step 8 — Merge Transfermarkt Features Into One Player Table

In [ ]:
# Start with the core players table
tm_master = players.copy()

# Join all feature tables on player_id
tm_master = tm_master.merge(latest_val,        on="player_id", how="left")
tm_master = tm_master.merge(peak_val,          on="player_id", how="left")
tm_master = tm_master.merge(transfer_summary,  on="player_id", how="left")
tm_master = tm_master.merge(career_stats,      on="player_id", how="left")
tm_master = tm_master.merge(recent_form,       on="player_id", how="left")
tm_master = tm_master.merge(
    club_info,
    left_on="current_club_id",
    right_on="club_id",
    how="left"
)

# Compute age from date of birth
tm_master["date_of_birth"] = pd.to_datetime(tm_master["date_of_birth"], errors="coerce")
tm_master["age"] = ((pd.Timestamp.now() - tm_master["date_of_birth"]).dt.days / 365.25).round(1)

print(f"✅ Transfermarkt master: {tm_master.shape[0]:,} players x {tm_master.shape[1]} columns")
print(f"   Columns: {list(tm_master.columns)}")

## 🔗 Step 9 — Merge FBref Stats + Transfermarkt into MASTER Scouting Dataset

In [ ]:
import unicodedata
import re

def normalize_name(name: str) -> str:
    """Normalize player names for fuzzy matching across sources."""
    if pd.isna(name):
        return ""
    # Unicode normalization (accents → base chars)
    name = unicodedata.normalize("NFD", str(name))
    name = "".join(c for c in name if unicodedata.category(c) != "Mn")
    # Lowercase, strip, remove special chars
    name = name.lower().strip()
    name = re.sub(r"[^a-z\s]", "", name)
    name = re.sub(r"\s+", " ", name)
    return name

# ── Normalize names in both sources ──
tm_master["name_norm"]    = tm_master["name"].apply(normalize_name)

# FBref player column name varies by source
player_col = "player" if "player" in fbref_master.columns else "Player"
fbref_master["name_norm"] = fbref_master[player_col].apply(normalize_name)

# ── Merge FBref + Transfermarkt on normalized name ──
scouting_master = fbref_master.merge(
    tm_master,
    on="name_norm",
    how="left",
    suffixes=("_fbref", "_tm")
)

# ── Match rate report ──
matched = scouting_master["player_id"].notna().sum()
total   = len(scouting_master)
print(f"✅ Total players in FBref:           {total:,}")
print(f"   Matched with Transfermarkt data: {matched:,} ({matched/total*100:.1f}%)")
print(f"   Unmatched (FBref only):          {total - matched:,}")
print(f"\n   Final shape: {scouting_master.shape[0]:,} rows x {scouting_master.shape[1]} columns")

## ⚙️ Step 10 — Feature Engineering (Scouting KPIs per 90 min)

In [ ]:
df = scouting_master.copy()

# Identify per-90 base column (minutes played from FBref standard stats)
# Column names may vary depending on scraping method
min_col = next((c for c in df.columns if "min" in c.lower() and "90" not in c.lower() and "standard" in c.lower()), None)
if min_col:
    df[min_col] = pd.to_numeric(df[min_col], errors="coerce").fillna(0)
    df["p90"] = df[min_col] / 90
else:
    # Fallback to career minutes
    df["p90"] = df["career_minutes"].fillna(0) / 90

def safe_per90(col, p90_col, df):
    return (pd.to_numeric(df[col], errors="coerce").fillna(0) / df[p90_col].replace(0, pd.NA)).round(3)

# ── Attacking Score (per 90) ──
# Goals + Assists + Key Passes
goal_col    = next((c for c in df.columns if "gls" in c.lower() or "goals" in c.lower() and "standard" in c.lower()), "career_goals")
ast_col     = next((c for c in df.columns if "ast" in c.lower() or "assists" in c.lower() and "standard" in c.lower()), "career_assists")
xg_col      = next((c for c in df.columns if "xg" in c.lower() and "standard" in c.lower()), None)
xa_col      = next((c for c in df.columns if "xa" in c.lower() or "xag" in c.lower()), None)

df["goals_p90"]   = safe_per90(goal_col, "p90", df)
df["assists_p90"] = safe_per90(ast_col,  "p90", df)
if xg_col:  df["xG_p90"] = safe_per90(xg_col, "p90", df)
if xa_col:  df["xA_p90"] = safe_per90(xa_col, "p90", df)

# ── Defensive Score (per 90) ──
tkl_col  = next((c for c in df.columns if "tkl" in c.lower() and "defense" in c.lower()), None)
int_col  = next((c for c in df.columns if "int" in c.lower()  and "defense" in c.lower()), None)
pres_col = next((c for c in df.columns if "press" in c.lower() and "defense" in c.lower()), None)

if tkl_col:  df["tackles_p90"]   = safe_per90(tkl_col,  "p90", df)
if int_col:  df["interceptions_p90"] = safe_per90(int_col, "p90", df)
if pres_col: df["pressures_p90"] = safe_per90(pres_col, "p90", df)

# ── Progression Score (per 90) ──
prog_carry = next((c for c in df.columns if "progc" in c.lower() or "progressive_carries" in c.lower()), None)
prog_pass  = next((c for c in df.columns if "progp" in c.lower() or "progressive_passes" in c.lower()), None)

if prog_carry: df["prog_carries_p90"] = safe_per90(prog_carry, "p90", df)
if prog_pass:  df["prog_passes_p90"]  = safe_per90(prog_pass,  "p90", df)

# ── Value Efficiency (performance vs market value) ──
df["value_efficiency"] = (
    (df["career_goals"].fillna(0) + df["career_assists"].fillna(0)) /
    (df["latest_market_value_eur"].replace(0, pd.NA) / 1_000_000)
).round(3)

print("✅ KPIs computed:")
kpi_cols = [c for c in df.columns if any(x in c for x in ["p90", "efficiency", "score"])]
print(f"   {kpi_cols}")

## 🧹 Step 11 — Final Cleaning & Export

In [ ]:
# ── Keep only players with meaningful data ──
# At least 90 minutes played to avoid noise from bench warmers
df_clean = df[df["p90"] >= 1.0].copy()

# ── Drop meta/duplicate columns ──
drop_cols = [c for c in df_clean.columns if any(x in c.lower() for x in [
    "name_norm", "url", "image", "_dup", "fbref_slug",
    "fbref_league_id", "player_code", "city_of_birth"
])]
df_clean.drop(columns=drop_cols, inplace=True, errors="ignore")

# ── Deduplicate (same player might appear if multiple table rows) ──
id_col = "player_id" if "player_id" in df_clean.columns else "name_norm"
df_clean.drop_duplicates(subset=[id_col], keep="first", inplace=True)

# ── Sort by market value descending ──
if "latest_market_value_eur" in df_clean.columns:
    df_clean.sort_values("latest_market_value_eur", ascending=False, inplace=True)

# ── Save master scouting dataset ──
df_clean.to_csv("master_scouting_dataset.csv", index=False)
print(f"✅ MASTER SCOUTING DATASET SAVED")
print(f"   Rows:    {df_clean.shape[0]:,} players")
print(f"   Columns: {df_clean.shape[1]}")
print(f"   File:    master_scouting_dataset.csv")
print(f"\n📊 Sample output:")
display_cols = [c for c in ["name", "age", "position", "league_name", "league_country",
                             "current_club_name", "latest_market_value_eur",
                             "goals_p90", "assists_p90", "value_efficiency"] if c in df_clean.columns]
print(df_clean[display_cols].head(10).to_string(index=False))

## 📊 Step 12 — Quick Validation & Stats

In [ ]:
import matplotlib.pyplot as plt

print("=" * 50)
print("MASTER DATASET SUMMARY")
print("=" * 50)

if "league_name" in df_clean.columns:
    league_counts = df_clean["league_name"].value_counts()
    print(f"\n📋 Players per league:")
    print(league_counts.to_string())

if "position" in df_clean.columns:
    print(f"\n📋 Players per position:")
    print(df_clean["position"].value_counts().to_string())

if "latest_market_value_eur" in df_clean.columns:
    vals = df_clean["latest_market_value_eur"].dropna()
    print(f"\n💶 Market value stats:")
    print(f"   Median: €{vals.median()/1e6:.1f}M")
    print(f"   Mean:   €{vals.mean()/1e6:.1f}M")
    print(f"   Max:    €{vals.max()/1e6:.0f}M")

# ── Completeness check ──
print(f"\n📉 Missing value rate per key column:")
key_check_cols = ["age", "position", "latest_market_value_eur",
                  "goals_p90", "assists_p90", "league_name"]
for c in key_check_cols:
    if c in df_clean.columns:
        miss = df_clean[c].isna().sum()
        pct  = miss / len(df_clean) * 100
        print(f"   {c:35s}: {miss:,} missing ({pct:.1f}%)")

---
## ✅ Pipeline Complete!

### Output file: `master_scouting_dataset.csv`

| Feature group | Source | Examples |
|---|---|---|
| Player identity | Transfermarkt Kaggle | name, age, nationality, position, foot |
| Market value | Transfermarkt Kaggle | current value, peak value, valuation history |
| Transfer history | Transfermarkt Kaggle | total transfers, fees paid, last move |
| Career stats | Transfermarkt Kaggle | career goals, assists, minutes, cards |
| Recent form | Transfermarkt Kaggle | last 12 months goals/assists/minutes |
| Real match stats | FBref (scraped) | xG, xA, progressive passes, pressures |
| Per-90 KPIs | Computed | goals/90, assists/90, tackles/90 |
| Efficiency | Computed | value_efficiency = G+A / market value |

### Next step: feed `master_scouting_dataset.csv` into your ML models
- **KNN Similarity Model** → find players similar to a reference
- **XGBoost Talent Score** → rank players 0-100
- **K-Means Clustering** → group by play style
---